# 01 — Prepare Assamese data

This downloads only Mozhi's `assamese` configuration. It never selects the Bengali configuration. For synthetic lines it uses only Assamese Wikipedia (`20231101.as`). The font filenames say Bengali because Unicode and font ecosystems use that name for the shared Eastern Nagari script; the rendered language content remains Assamese.

In [ ]:
from PIL import features

from axomiya_ocr.data.mozhi import prepare_mozhi

RUN_FULL = True  # False makes a 256-example pipeline smoke test
report = prepare_mozhi(
    "data/processed/mozhi_assamese",
    max_per_split=None if RUN_FULL else 256,
    num_proc=4,
    overwrite=True,
)
report["dataset"], report["config"], report["revision"]

In [ ]:
import json

print(
    json.dumps(
        {
            "splits": {name: info["after_filter_and_deduplication"] for name, info in report["splits"].items()},
            "with_ra": {name: info["with_ra"] for name, info in report["splits"].items()},
            "with_wa": {name: info["with_wa"] for name, info in report["splits"].items()},
            "unseen_characters": report["unseen_characters"],
            "split_leakage_before_cleanup": report["split_leakage_before_cleanup"],
            "split_leakage_after_cleanup": report["split_leakage_after_cleanup"],
        },
        ensure_ascii=False,
        indent=2,
    )
)
assert report["config"] == "assamese"
assert not report["unseen_characters"]["validation"]
assert not report["unseen_characters"]["test"]

## Optional synthetic Assamese lines

Run these cells for the production training set. Complex-script shaping is mandatory; the code stops if Pillow lacks RAQM rather than generating malformed conjuncts. Standard Linux notebook runtimes normally include it.

In [ ]:
print("RAQM shaping available:", features.check_feature("raqm"))
if not features.check_feature("raqm"):
    raise RuntimeError("Use a Linux notebook image with Pillow RAQM support.")
!python scripts/download_fonts.py
!python scripts/prepare_corpus.py --max-lines 250000
!python scripts/render_synthetic.py --samples 250000 --overwrite

In [ ]:
import matplotlib.pyplot as plt
from datasets import load_from_disk

synthetic = load_from_disk("data/processed/synthetic_assamese")["train"]
fig, axes = plt.subplots(3, 1, figsize=(14, 5))
for axis, example in zip(axes, synthetic.select(range(3)), strict=True):
    axis.imshow(example["image"], cmap="gray")
    axis.set_title(example["text"])
    axis.axis("off")
plt.tight_layout()